In [ ]:
#| default_exp convert

# Converting Python into notebooks

> Move Python source into numbered nbdev notebooks without losing source order.

#| export
Conversion is a bridge for projects that began as `.py` files but want nbdev notebooks as the source of truth. `py2nb` preserves source order and creates explicit export cells. `convert` selects file, folder, or project migration, while `py2nbdev` adds an nbdev scaffold. Each result is a draft notebook that still needs the rationale and examples a reader would expect.

In [ ]:
#| exportd
from nbskill.convert import py2nb, convert, py2nbdev
from nbskill.foundation import demo_path, remove_demo_path, demo_path_context, write_demo_file
from fastcore.nbio import read_nb
from fastcore.test import test_eq

## Production contract

Conversion is Python-first. `convert` migrates one Python file, a folder of Python files, or a pure-Python package into an nbdev project. The lower-level helpers preserve source order, create stable nbdev cells and ids, support dry-run/force controls, keep package tree choices explicit, and export generated modules when writing real notebooks.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from nbdev.export import nb_export
from runpy import run_path
from nbskill.convert import py2nb as _example_py2nb
from nbskill.foundation import write_demo_notebook
from fastcore.test import *
import tomllib

In [ ]:
#| export
import ast,copy
import fnmatch, hashlib, json, os, subprocess, sys
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb

from nbskill.foundation import api_return, numbered_notebook_path

## Slicing Python source with `ast`

Conversion starts by finding exact source ranges for imports, functions, classes, and methods. Using `ast` keeps the conversion structured instead of relying on fragile string searches.

In [ ]:
#| exporti
#| hide
def _node_source(lines, node):
    source = ast.get_source_segment("\n".join(lines), node)
    decorators = getattr(node, "decorator_list", [])
    if not decorators: return source
    start = min(decorator.lineno for decorator in decorators) - 1
    return "\n".join([*lines[start:node.lineno - 1], source])

In [ ]:
#| exporti
#| hide
def _temporary_assignment_target(target, bindings, reserved_names):
    if isinstance(target, (ast.Tuple, ast.List)):
        target.elts = [_temporary_assignment_target(item, bindings, reserved_names) for item in target.elts]
        return target
    if isinstance(target, ast.Starred):
        target.value = _temporary_assignment_target(target.value, bindings, reserved_names)
        return target
    name = f"_nbskill_value_{len(bindings)}"
    while name in reserved_names: name += "_"
    bindings.append((target, name))
    return ast.Name(id=name, ctx=ast.Store())

In [ ]:
#| exporti
#| hide
def _assignment_sources(node, lines, reserved_names):
    if not isinstance(node, ast.Assign): return [_node_source(lines, node)]
    assignment,bindings = copy.deepcopy(node),[]
    assignment.targets = [_temporary_assignment_target(target, bindings, reserved_names) for target in assignment.targets]
    if len(bindings) == 1: return [_node_source(lines, node)]
    sources = [ast.unparse(ast.fix_missing_locations(ast.Assign(targets=[target], value=ast.Name(id=name, ctx=ast.Load()))))
        for target, name in bindings]
    sources[0] = f"{ast.unparse(ast.fix_missing_locations(assignment))}\n{sources[0]}"
    sources[-1] += f"\ndel {', '.join(name for _, name in bindings)}"
    return sources

Every method gets its own exported cell, including constructors, async methods, and methods without a receiver. Methods use `patch_to` so their receiver annotations stay intact. The converter makes zero-argument `super()`, private names, and class-local defaults explicit because the method now lives outside the class body.

In [ ]:
#| exporti
#| hide
def _patchable_method(node): return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))

In [ ]:
ordinary_method = ast.parse("def area(self): return self.width").body[0]
decorated_method = ast.parse("@property\ndef area(self): return self.width").body[0]
receiverless_method = ast.parse("def area(): return 1").body[0]
assert all(_patchable_method(node) for node in (ordinary_method, decorated_method, receiverless_method))
[_patchable_method(node) for node in (ordinary_method, decorated_method, receiverless_method)]

In [ ]:
#| exporti
#| hide
def _mangled_name(name, class_name):
    if name.startswith("__") and not name.endswith("__") and class_name.strip("_"): return f"_{class_name.lstrip('_')}{name}"
    return name

In [ ]:
#| exporti
#| hide
class _MethodContext(ast.NodeTransformer):
    def __init__(self, class_name, receiver):
        self.class_name,self.receiver = class_name,_mangled_name(receiver, class_name) if receiver else None

    def visit_Call(self, node):
        node = self.generic_visit(node)
        if self.receiver and isinstance(node.func, ast.Name) and node.func.id == "super" and not node.args and not node.keywords:
            node.args = [ast.Name(id=self.class_name, ctx=ast.Load()), ast.Name(id=self.receiver, ctx=ast.Load())]
        return node

    def visit_Name(self, node):
        name = self.class_name if node.id == "__class__" else _mangled_name(node.id, self.class_name)
        return ast.copy_location(ast.Name(id=name, ctx=node.ctx), node)

    def visit_Attribute(self, node):
        node = self.generic_visit(node)
        node.attr = _mangled_name(node.attr, self.class_name)
        return node

    def visit_arg(self, node):
        node = self.generic_visit(node)
        node.arg = _mangled_name(node.arg, self.class_name)
        return node

    def visit_FunctionDef(self, node):
        node = self.generic_visit(node)
        node.name = _mangled_name(node.name, self.class_name)
        return node

    def visit_AsyncFunctionDef(self, node): return self.visit_FunctionDef(node)
    def visit_ClassDef(self, node): return node

In [ ]:
#| exporti
#| hide
class _ClassReference(ast.NodeTransformer):
    def __init__(self, class_name, names): self.class_name,self.names = class_name,names

    def visit_Name(self, node):
        if node.id in self.names:
            attr = _mangled_name(node.id, self.class_name)
            return ast.copy_location(ast.Attribute(value=ast.Name(id=self.class_name, ctx=ast.Load()), attr=attr, ctx=node.ctx), node)
        return node

In [ ]:
#| exporti
#| hide
def _decorated_method_source(source, method, class_name, reserved_names, class_names):
    temporary = "_nbskill_decorators"
    while temporary in reserved_names: temporary += "_"
    expressions = ", ".join(ast.unparse(_ClassReference(class_name, class_names).visit(copy.deepcopy(decorator)))
        for decorator in method.decorator_list)
    header = f"{temporary} = ({expressions},)"
    name = _mangled_name(method.name, class_name)
    descriptor = f"{class_name}.__dict__[{name!r}]"
    value = descriptor
    for idx in reversed(range(len(method.decorator_list))): value = f"{temporary}[{idx}]({value})"
    set_name = f"if hasattr({descriptor}, '__set_name__'): {descriptor}.__set_name__({class_name}, {name!r})"
    return f"{header}\n{source}\n{class_name}.{name} = {value}\n{set_name}\ndel {temporary}"

In [ ]:
#| exporti
#| hide
def _patch_method_source(method, class_name, reserved_names=(), class_names=()):
    node = copy.deepcopy(method)
    node.decorator_list = []
    qualifier = _ClassReference(class_name, class_names)
    node.args = qualifier.visit(node.args)
    if node.returns is not None: node.returns = qualifier.visit(node.returns)
    args = node.args.posonlyargs + node.args.args
    node = _MethodContext(class_name, args[0].arg if args else None).visit(node)
    ast.fix_missing_locations(node)
    source = f"@patch_to({class_name})\n{ast.unparse(node)}"
    original = f"_orig_{node.name}"
    if original not in reserved_names: source += f"\nif {original!r} in {class_name}.__dict__: delattr({class_name}, {original!r})"
    if method.decorator_list: return _decorated_method_source(source, method, class_name, reserved_names, class_names)
    return source

The class keeps its attributes and gets `pass` when removing its methods leaves an empty body. Method decorators are captured before replacement, so property setters can still refer to their getter. Descriptors receive `__set_name__`; class decorators run after all methods are attached. Synthetic `_orig_` aliases from `patch_to` are removed before abstract methods are recalculated, since an alias of an abstract method would otherwise prevent instantiation.

In [ ]:
#| exporti
#| hide
def _class_without_methods(lines, cls, methods):
    class_start = cls.lineno
    src_lines = lines[class_start - 1:cls.end_lineno]
    remove_ranges = []
    for method in methods:
        start = min([method.lineno, *[d.lineno for d in method.decorator_list]]) - class_start
        stop = method.end_lineno - class_start + 1
        remove_ranges.append((start, stop))
    for start, stop in sorted(remove_ranges, reverse=True): del src_lines[start:stop]
    if not any(child not in methods for child in cls.body): src_lines.append("    pass")
    return "\n".join(src_lines).strip("\n")

In [ ]:
#| exporti
#| hide
def _class_cells(lines, cls, reserved_names):
    methods = [child for child in cls.body if _patchable_method(child)]
    class_names = {child.name for child in cls.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
    for child in cls.body:
        targets = child.targets if isinstance(child, ast.Assign) else [child.target] if isinstance(child, ast.AnnAssign) else []
        class_names.update(node.id for target in targets for node in ast.walk(target) if isinstance(node, ast.Name))
    sources = [_class_without_methods(lines, cls, methods)]
    sources.extend(_patch_method_source(method, cls.name, reserved_names, class_names) for method in methods)
    for decorator in reversed(cls.decorator_list): sources[-1] += f"\n{cls.name} = ({ast.unparse(decorator)})({cls.name})"
    if methods: sources[-1] += f"\nupdate_abstractmethods({cls.name})"
    return [_export_cell(source) for source in sources]

In [ ]:
#| exporti
#| hide
def _export_cell(source):
    return mk_cell(f"#| export\n{source.strip()}")

## Assembling nbdev cells

The converter writes `#| default_exp` first and keeps imports together in one cell. Each module-level constant, function, and class gets its own `#| export` cell, and every method is exported separately, regardless of its length. Other module statements keep their source order; existing `__all__` declarations are omitted. Chained assignments and unpacking use temporary bindings so each constant gets a cell without evaluating the right-hand side twice.

In [ ]:
#| exporti
#| hide
def _stable_cell_id(default_exp, idx, source):
    raw = f"{default_exp}:{idx}:{source}".encode("utf-8")
    return f"py2nb-{hashlib.sha1(raw).hexdigest()[:8]}"

In [ ]:
#| exporti
#| hide
def _stabilize_cells(cells, default_exp):
    for idx, cell in enumerate(cells): cell.id = _stable_cell_id(default_exp, idx, getattr(cell, "source", ""))
    return cells

A module docstring is an expression containing literal text. It can become notebook Markdown without confusing ordinary assignments with documentation.

In [ ]:
#| exporti
#| hide
def _is_module_docstring(node): return isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str)

In [ ]:
doc_tree = ast.parse('"Module purpose"\nmessage = "text"')
assert _is_module_docstring(doc_tree.body[0]) and not _is_module_docstring(doc_tree.body[1])
[_is_module_docstring(node) for node in doc_tree.body]

In [ ]:
#| exporti
#| hide
def _py2nb_cells(source, default_exp, class_lines=100, method_lines=10):
    tree = ast.parse(source)
    lines = source.splitlines()
    cells = [mk_cell(f"#| default_exp {default_exp}")]
    body = list(tree.body)
    if body and _is_module_docstring(body[0]): cells.append(mk_cell(body.pop(0).value.value.strip(), cell_type="markdown"))
    imports = [_node_source(lines, node) for node in body if isinstance(node, (ast.Import, ast.ImportFrom))]
    if any(isinstance(node, ast.ClassDef) and any(_patchable_method(child) for child in node.body) for node in body):
        imports.extend(["from fastcore.basics import patch_to", "from abc import update_abstractmethods"])
    if imports: cells.append(_export_cell("\n".join(imports)))
    reserved_names = {node.id for node in ast.walk(tree) if isinstance(node, ast.Name)}
    reserved_names.update(node.name for node in ast.walk(tree) if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    for node in body:
        if isinstance(node, (ast.Import, ast.ImportFrom)): continue
        targets = node.targets if isinstance(node, ast.Assign) else [node.target] if isinstance(node, ast.AnnAssign) else []
        if any(isinstance(target, ast.Name) and target.id == "__all__" for target in targets): continue
        if isinstance(node, ast.ClassDef): cells.extend(_class_cells(lines, node, reserved_names))
        else: cells.extend(_export_cell(source) for source in _assignment_sources(node, lines, reserved_names))
    return _stabilize_cells(cells, default_exp)

In [ ]:
#| hide
granular_source = r"""from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass
from functools import cached_property
FIRST = 1
SECOND: int = 2
class Parent(ABC):
    __offset = FIRST
    @abstractmethod
    def value(self): return self.__offset
@dataclass
class Child(Parent):
    ZERO = 0
    def __init__(self): self.__value = SECOND
    def value(self): return super().value() + self.__value
    @staticmethod
    def zero(value=ZERO): return value
    @classmethod
    def make(klass): return klass()
    @property
    def size(self): return self.__value
    @size.setter
    def size(self, value): self.__value = value
    @cached_property
    def cached(self): return self.size * 2
    async def ready(self): return self.size
def answer(): return Child().value()
THIRD = answer()
"""

In [ ]:
#| hide
granular_cells = _py2nb_cells(granular_source, "granular")
granular_code = [cell.source for cell in granular_cells[1:] if cell.cell_type == "code"]
granular_trees = [ast.parse(source) for source in granular_code]
assert all(source.startswith("#| export\n") for source in granular_code)
test_eq(sum("FIRST = 1" in source or "SECOND: int = 2" in source for source in granular_code), 2)
assert all(not any(isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) for child in node.body)
    for tree in granular_trees for node in tree.body if isinstance(node, ast.ClassDef))
test_eq(sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    for tree in granular_trees for node in tree.body), 10)

In [ ]:
#| hide
assignment_source = r"""calls = []
def make_value(): calls.append(1); return [10, 20, 30]
LEFT = RIGHT = make_value()
HEAD, *TAIL = LEFT
FIRST = 1; SECOND = 2
"""
assignment_cells = _py2nb_cells(assignment_source, "assignments")[1:]
assignment_names = []
for cell in assignment_cells:
    names = {node.id for node in ast.walk(ast.parse(cell.source))
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store) and node.id.isupper()}
    assignment_names.append(names)
assert all(len(names) <= 1 for names in assignment_names)

In [ ]:
#| hide
assignment_module = "\n".join(cell.source for cell in assignment_cells)
with write_demo_file("assignment_cells.py", assignment_module) as path: assignment_values = run_path(path)
test_eq(assignment_values["calls"], [1])
test_is(assignment_values["LEFT"], assignment_values["RIGHT"])
test_eq((assignment_values["HEAD"], assignment_values["TAIL"], assignment_values["FIRST"], assignment_values["SECOND"]),
    (10, [20, 30], 1, 2))

In [ ]:
#| exporti
#| hide
def _module_name_for_path(path, module_base=None): return Path(path).stem

Single-file conversion reads Python source, derives a numbered destination and prepares notebook cells before writing. Dry runs return the same destination plan without creating it.

In [ ]:
#| exporti
#| hide
def _py2nb_file(path, nbs_path="nbs", dest=None, class_lines=100, method_lines=10, default_exp=None, dry_run=False, force=True, number=None):
    pth = Path(path)
    source = pth.read_text(encoding="utf-8")
    default_exp = default_exp or pth.stem
    out_root = Path(dest) if dest and not Path(dest).suffix else Path(dest).parent if dest else Path(nbs_path)
    out_path = numbered_notebook_path(out_root, default_exp, index=number)
    if out_path.exists() and not force: raise FileExistsError(f"Refusing to overwrite existing notebook: {out_path}")
    nb = new_nb(_py2nb_cells(source, default_exp, class_lines=class_lines, method_lines=method_lines))
    if not dry_run:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        write_nb(nb, out_path)
    return dict(source=str(pth), notebook=str(out_path), default_exp=default_exp, cell_count=len(nb.cells), dry_run=dry_run)

## Public conversion command

`convert` dispatches by mode. `mode="notebook"` handles either one Python file or a folder of Python files. `mode="project"` creates a pragmatic nbdev project from a pure-Python package.

In [ ]:
#| exporti
#| hide
_DEFAULT_IGNORE_DIRS = {'__pycache__', '.git', '.hg', '.mypy_cache', '.pytest_cache', '.ruff_cache', '.tox', '.venv', 'venv', 'env',
    'build', 'dist', '.ipynb_checkpoints', 'node_modules'}

In [ ]:
#| exporti
#| hide
_DEFAULT_IGNORE_FILES = {"__init__.py"}

Conversion skips generated and version-control directories by default. Test directories are classified separately so callers can opt into test conversion.

In [ ]:
#| exporti
#| hide
_TEST_DIRS = {"test", "tests", "testing"}

In [ ]:
#| exporti
#| hide
def _patterns(value):
    if not value: return []
    if isinstance(value, (list, tuple, set)): return [str(item) for item in value if str(item)]
    return [item.strip() for item in str(value).split(",") if item.strip()]

Source filters match either the relative path or the filename. This supports both directory-specific and file-wide exclusions.

In [ ]:
#| exporti
#| hide
def _matches_any(path, patterns):
    text = path.as_posix()
    return any(fnmatch.fnmatch(text, pat) or fnmatch.fnmatch(path.name, pat) for pat in patterns)

In [ ]:
candidate_path = Path("package/tests/test_parse.py")
assert _matches_any(candidate_path, ["tests/*", "test_*.py"])
assert not _matches_any(candidate_path, ["*.ipynb"])
[_matches_any(candidate_path, patterns) for patterns in (["test_*.py"], ["*.ipynb"])]

In [ ]:
#| exporti
#| hide
def _package_dirs(root):
    return sorted((path for path in root.iterdir() if path.is_dir() and (path / '__init__.py').exists() and (path.name not in _DEFAULT_IGNORE_DIRS)))

In [ ]:
#| exporti
#| hide
def _resolve_source_tree(path, package=None):
    root = Path(path)
    if root.is_file(): return root, root.parent, root.parent
    if package:
        for candidate in (root / package, root / "src" / package, root):
            if candidate.exists() and candidate.name == package: return candidate, candidate.parent, root
    if root.name == "src" and _package_dirs(root):
        pkgs = _package_dirs(root)
        return (pkgs[0], root, root) if len(pkgs) == 1 else (root, root, root)
    if (root / "src").is_dir() and _package_dirs(root / "src"):
        pkgs = _package_dirs(root / "src")
        return (pkgs[0], root / "src", root) if len(pkgs) == 1 else (root / "src", root / "src", root)
    if (root / "__init__.py").exists(): return root, root.parent, root
    pkgs = _package_dirs(root)
    if len(pkgs) == 1: return pkgs[0], root, root
    return root, root, root

In [ ]:
#| exporti
#| hide
def _is_ignored_py(path, rel, include=None, exclude=None, skip_init=True, include_tests=False):
    if any(part in _DEFAULT_IGNORE_DIRS or part.startswith(".") for part in rel.parts[:-1]): return True
    if skip_init and path.name in _DEFAULT_IGNORE_FILES: return True
    if not include_tests and (set(rel.parts[:-1]) & _TEST_DIRS or path.name.startswith("test_") or path.name.endswith("_test.py")): return True
    if include and not _matches_any(rel, include): return True
    if exclude and _matches_any(rel, exclude): return True
    return False

In [ ]:
#| exporti
#| hide
def _python_files(scan_root, recursive=True, maxdepth=None):
    files = scan_root.rglob("*.py") if recursive else scan_root.glob("*.py")
    for path in sorted(files):
        rel = path.relative_to(scan_root)
        if maxdepth is not None and len(rel.parts) - 1 > maxdepth: continue
        yield path

In [ ]:
#| exporti
#| hide
def _py2nbs_folder(path, nbs_path='nbs', recursive=True, maxdepth=None, preserve_tree=False, class_lines=100, method_lines=10, package=None,
    include=None, exclude=None, skip_init=True, include_tests=False, dry_run=False, force=True):
    if preserve_tree: raise ValueError("Notebook conversion is flat; preserve_tree must be False")
    scan_root, module_base, project_root = _resolve_source_tree(path, package=package)
    include, exclude = _patterns(include), _patterns(exclude)
    report = dict(source_root=str(project_root), scan_root=str(scan_root), module_base=str(module_base), nbs_path=str(nbs_path),
        converted=[], skipped=[], blockers=[], dry_run=dry_run)
    seen_modules = {}
    number = None
    for pth in _python_files(scan_root, recursive=recursive, maxdepth=maxdepth):
        rel = pth.relative_to(scan_root)
        if _is_ignored_py(pth, rel, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests):
            report["skipped"].append(str(pth))
            continue
        default_exp = _module_name_for_path(pth, module_base=module_base)
        if default_exp in seen_modules:
            report["blockers"].append(f"Module collision: {pth} and {seen_modules[default_exp]} -> {default_exp}")
            continue
        seen_modules[default_exp] = str(pth)
        if number is None: number = int(numbered_notebook_path(nbs_path, default_exp).name.split("_", 1)[0])
        item = _py2nb_file(str(pth), nbs_path=nbs_path, class_lines=class_lines, method_lines=method_lines, default_exp=default_exp,
            dry_run=dry_run, force=force, number=number)
        number += 1
        report["converted"].append(item)
        print(f"{'Would write' if dry_run else 'Wrote'} {item['cell_count']} cells to {item['notebook']}")
    print(f"{'Would convert' if dry_run else 'Converted'} {len(report['converted'])} Python files to {nbs_path}")
    if report["skipped"]: print(f"Skipped {len(report['skipped'])} ignored Python files")
    if report["blockers"]: print(f"Blocked {len(report['blockers'])} conversion item(s)")
    return report

#| export
### `py2nb`

`py2nb` converts a file or package into flat numbered notebooks. Each constant, function, class, and method gets a separate `#| export` cell; imports share one cell. The legacy `class_lines` and `method_lines` arguments are accepted for compatibility and no longer affect the split. Conversion supplies a starting point; maintainers still add the domain rationale, examples, and assertions. A real Python input produces an nbdev export target and one exported function cell:

In [ ]:
#| exporti
def py2nb(
    path: str, # Python file or folder to convert
    nbs_path: str = "nbs", # Folder for generated notebooks
    dest: str | None = None, # Output folder for generated notebooks
    recursive: bool = True, # Search subfolders when path is a folder
    maxdepth: int | None = None, # Maximum folder depth to search
    preserve_tree: bool = False, # Must remain false because notebooks are flat
    class_lines: int = 100, # Compatibility only; all methods are split
    method_lines: int = 10, # Compatibility only; all methods are split
    package: str | None = None, # Package name to select inside a project root
    include: str | None = None, # Comma-separated include glob(s)
    exclude: str | None = None, # Comma-separated exclude glob(s)
    skip_init: bool = True, # Skip __init__.py by default
    include_tests: bool = False, # Convert test files too
    dry_run: bool = False, # Plan conversion without writing notebooks
    force: bool = True, # Overwrite existing generated notebooks
):
    "Convert Python files into flat numbered nbdev notebooks using AST parsing."
    pth = Path(path)
    if pth.is_dir():
        report = _py2nbs_folder(pth, nbs_path=dest or nbs_path, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree,
            class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init,
            include_tests=include_tests, dry_run=dry_run, force=force)
        return api_return(report)
    report = _py2nb_file(path, nbs_path=nbs_path, dest=dest, class_lines=class_lines, method_lines=method_lines, default_exp=pth.stem,
        dry_run=dry_run, force=force)
    print(f"{'Would write' if dry_run else 'Wrote'} {report['cell_count']} cells to {report['notebook']}")
    return api_return(report)

In [ ]:
#| exportd
with demo_path_context("05_convert_usage") as root:
    root.mkdir()
    source_path = root / "calc.py"
    source_path.write_text("def add(a, b):\n    return a + b\n", encoding="utf-8")
    conversion = py2nb(source_path, nbs_path=root / "nbs")
    converted = read_nb(conversion["notebook"])
    test_eq(converted.cells[0].source, "#| default_exp calc")
    assert any("def add(a, b)" in cell.source for cell in converted.cells)
    conversion_preview = dict(cell_count=conversion["cell_count"], export_target=converted.cells[0].source)
conversion_preview

Wrote 2 cells to /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/nbskill/05_convert_usage/nbs/00_calc.ipynb


{'cell_count': 2, 'export_target': '#| default_exp calc'}

In [ ]:
#| hide

root = demo_path("05_convert_tests")
try:
    root.mkdir()
    src = root / "demo.py"
    nbs_path = root / "nbs"
    src.write_text("import math\nFIRST = 1\ndef add(a, b):\n    return a + b\nfrom pathlib import Path\nSECOND = 2\ndef sub(a, b):\n    return a - b\n", encoding="utf-8")
    py2nb(str(src), nbs_path=str(nbs_path))
    nb = read_nb(nbs_path / "00_demo.ipynb")
    code_cells = [cell.source for cell in nb.cells if cell.cell_type == "code"]
    function_cells = [cell for cell in code_cells if "def " in cell]

    pkg = root / "pkg"
    (pkg / "sub").mkdir(parents=True)
    (pkg / "one.py").write_text("def one():\n    return 1\n", encoding="utf-8")
    (pkg / "sub" / "two.py").write_text("def two():\n    return 2\n", encoding="utf-8")
    all_nbs = root / "all_nbs"
    py2nb(str(pkg), nbs_path=str(all_nbs))
    test_eq(sorted(path.name for path in all_nbs.glob("*.ipynb")), ["00_one.ipynb", "01_two.ipynb"])
    assert not any(path.is_dir() for path in all_nbs.iterdir())
    test_eq(read_nb(all_nbs / "00_one.ipynb").cells[0].source, "#| default_exp one")
    test_eq(read_nb(all_nbs / "01_two.ipynb").cells[0].source, "#| default_exp two")

finally: remove_demo_path(root)

In [ ]:
#| hide
test_eq(nb.cells[0].source, "#| default_exp demo")
test_eq(sum("import " in cell for cell in code_cells), 1)
test_eq(sum("FIRST = 1" in cell or "SECOND = 2" in cell for cell in code_cells), 2)
test_eq(len(function_cells), 2)
assert all(cell.startswith("#| export\n") for cell in function_cells)

In [ ]:
#| hide
with demo_path_context("05_convert_export") as root:
    root.mkdir()
    source = root / "sample.py"
    source.write_text(granular_source, encoding="utf-8")
    result = py2nb(source, nbs_path=root / "nbs")
    notebook = read_nb(result["notebook"])
    exported_cells = [cell for cell in notebook.cells if cell.cell_type == "code"
        and not cell.source.startswith("#| default_exp")]
    assert all(cell.source.startswith("#| export\n") for cell in exported_cells)
    assert any("@patch" in cell.source for cell in exported_cells)
    nb_export(result["notebook"], lib_path=root / "exported", solo_nb=True)
    module = run_path(root / "exported" / "sample.py")
    test_eq(module["answer"](), module["THIRD"])
test_eq(module["THIRD"], 3)

In [ ]:
#| hide
child = module["Child"].make()
test_eq((child.value(), child.zero(), child.size, child.cached), (3, 0, 2, 4))
child.size = 8
test_eq((child.value(), child.size, child.cached), (9, 8, 4))
test_eq(await child.ready(), 8)
test_fail(module["Parent"])

#| export
### `convert`

`convert` selects notebook conversion or project creation through one entry point. Notebook output is flat, so pass `preserve_tree=False` when converting a directory. This dispatch example plans a conversion and leaves the source file untouched:

In [ ]:
#| export
def convert(
    path: str,  # Python file/folder, or existing Python project/package root
    mode: str = "notebook",  # notebook for file/folder conversion, project for nbdev project creation
    dest: str | None = None,  # Notebook path/output folder for notebook mode; project root for project mode
    nbs_path: str = "nbs",  # Destination notebooks folder
    recursive: bool = True,  # Search subfolders in notebook mode
    maxdepth: int | None = None,  # Maximum folder scan depth in notebook mode
    preserve_tree: bool = True,  # Preserve package folder structure under nbs_path
    class_lines: int = 100,  # Compatibility only; all methods are split
    method_lines: int = 10,  # Compatibility only; all methods are split
    package: str | None = None,  # Package root name
    include: str | None = None,  # Comma-separated include globs
    exclude: str | None = None,  # Comma-separated exclude globs
    skip_init: bool = True,  # Skip __init__.py files in notebook mode
    include_tests: bool = False,  # Include test files in notebook mode
    dry_run: bool = False,  # Show planned writes without writing
    force: bool = True,  # Overwrite existing notebooks/files
    run_validation: bool = True,  # Run nbdev validation in project mode
):
    "Convert Python sources to nbdev notebooks or a small nbdev project."
    if mode == "notebook":
        return py2nb(path, nbs_path=nbs_path, dest=dest, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree,
            class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init,
            include_tests=include_tests, dry_run=dry_run, force=force)
    if mode == "project":
        if dest is None: raise ValueError("convert(mode='project') needs dest")
        return py2nbdev(path, dest, package=package, nbs_path=nbs_path, dry_run=dry_run, force=force, run_validation=run_validation)
    raise ValueError("mode must be 'notebook' or 'project'")

In [ ]:
#| exportd
with write_demo_file("operation_total.py", "def total(values):\n    return sum(values)\n") as source_path:
    destination = demo_path("05_convert_plan.ipynb")
    planned = convert(source_path, dest=destination, preserve_tree=False, dry_run=True)
    assert planned["dry_run"] and not destination.exists()
    plan_preview = dict(dry_run=planned["dry_run"], cell_count=planned["cell_count"])
plan_preview

Would write 2 cells to /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/nbskill/00_operation_total.ipynb


{'dry_run': True, 'cell_count': 2}

In [ ]:
#| exporti
#| hide
def _primary_package_name(source, package=None):
    scan_root, module_base, _ = _resolve_source_tree(source, package=package)
    if package: return package
    if (scan_root / "__init__.py").exists(): return scan_root.name
    pkgs = _package_dirs(scan_root)
    if len(pkgs) == 1: return pkgs[0].name
    return Path(source).resolve().name.replace("-", "_")

In [ ]:
#| exporti
#| hide
def _write_if_allowed(path, text, dry_run=True, force=False):
    pth = Path(path)
    if dry_run: return {"path": str(pth), "action": "would-write"}
    if pth.exists() and not force: return {"path": str(pth), "action": "exists"}
    pth.parent.mkdir(parents=True, exist_ok=True)
    pth.write_text(text, encoding="utf-8")
    return {"path": str(pth), "action": "wrote"}

Project conversion plans destination files before writing. Package naming, overwrite checks and scaffold configuration all use the selected project root.

In [ ]:
#| exporti
#| hide
def _nbdev_pyproject_text(name, package, nbs_path):
    return f'''[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "{name}"
dynamic = ["version"]
description = "Converted nbdev project"
requires-python = ">=3.10"
dependencies = ["nbdev>=3.0.15"]

[project.entry-points.nbdev]
{package} = "{package}._modidx:d"

[tool.setuptools.dynamic]
version = {{attr = "{package}.__version__"}}

[tool.setuptools.packages.find]
include = ["{package}*"]

[tool.nbdev]
nbs_path = "{nbs_path}"
lib_path = "{package}"
doc_path = "_docs"
'''

In [ ]:
#| exporti
#| hide
def _validate_nbdev_project(dest, package, run_validation=True, dry_run=True):
    result = dict(requested=run_validation, export=None, imports=[], errors=[])
    if dry_run or not run_validation: return result
    try:
        from nbdev.export import nb_export
        exported = []
        for nb_path in sorted((Path(dest) / "nbs").rglob("*.ipynb")):
            nb_export(str(nb_path), str(dest))
            exported.append(str(nb_path))
        result["export"] = dict(returncode=0, notebooks=exported, stdout="", stderr="")
    except BaseException as exc:
        result["export"] = dict(returncode=1, stdout="", stderr=f"{type(exc).__name__}: {exc}")
        result["errors"].append("nbdev export failed")
        return result
    env = dict(**os.environ, PYTHONPATH=str(Path(dest).resolve()))
    proc = subprocess.run([sys.executable, "-c", f"import {package}"], cwd=dest, text=True, capture_output=True, env=env)
    result["imports"].append(dict(module=package, returncode=proc.returncode, stderr=proc.stderr[-4000:]))
    if proc.returncode: result["errors"].append(f"import {package} failed")
    return result

## Creating a project scaffold

#| export
### `py2nbdev`

`py2nbdev` adds the nbdev project configuration, version module, and index notebook needed around converted source. It defaults to `dry_run=True` so callers can inspect every planned file before creating a project. Validation runs after an actual write unless `run_validation=False` is requested:

In [ ]:
#| exporti
def py2nbdev(
    source: str, # Existing Python project/package root
    dest: str, # Destination nbdev project root
    package: str | None = None, # Package name to convert
    nbs_path: str = "nbs", # Notebook folder inside dest
    dry_run: bool = True, # Plan without writing by default
    force: bool = False, # Overwrite existing scaffold/notebooks
    run_validation: bool = True, # Run nbdev export and import checks after writing
):
    "Create a small nbdev project from a pure-Python package."
    package_name = _primary_package_name(source, package=package)
    dest_root = Path(dest)
    notebook_root = dest_root / nbs_path
    scaffold = [
        _write_if_allowed(dest_root / "pyproject.toml", _nbdev_pyproject_text(package_name, package_name, nbs_path), dry_run=dry_run, force=force),
        _write_if_allowed(dest_root / package_name / "__init__.py", '__version__ = "0.0.1"\n', dry_run=dry_run, force=force),
        _write_if_allowed(dest_root / nbs_path / "index.ipynb", json.dumps(new_nb([mk_cell(f"# {package_name}", cell_type="markdown")]), indent=2), dry_run=dry_run, force=force)]
    conversion = _py2nbs_folder(source, nbs_path=str(notebook_root), package=package_name, dry_run=dry_run, force=force, skip_init=True,
        include_tests=False)
    validation = _validate_nbdev_project(dest_root, package_name, run_validation=run_validation, dry_run=dry_run)
    report = dict(source=str(source), dest=str(dest_root), package=package_name, dry_run=dry_run, scaffold=scaffold, conversion=conversion,
        validation=validation, blockers=conversion.get('blockers', []) + validation.get('errors', []))
    print(json.dumps(report, indent=2, sort_keys=True, default=str))
    return api_return(report)

In [ ]:
#| exportd
root = demo_path("05_convert_scaffold_preview")
try:
    package = root / "calc"
    package.mkdir(parents=True)
    (package / "core.py").write_text("def answer():\n    return 42\n", encoding="utf-8")
    scaffold = py2nbdev(package, root / "nbdev_calc")
    assert scaffold["dry_run"] and not (root / "nbdev_calc").exists()
    scaffold_preview = dict(dry_run=scaffold["dry_run"], package=scaffold["package"])
finally: remove_demo_path(root)
scaffold_preview

In [ ]:
#| hide
root = demo_path("05_convert_project_tests")
try:
    pkg = root / "pkg"
    pkg.mkdir(parents=True)
    (pkg / "one.py").write_text("def one():\n    return 1\n", encoding="utf-8")
    project = root / "project"
    py2nbdev(str(pkg), str(project), dry_run=False, run_validation=False)
    config = tomllib.loads((project / "pyproject.toml").read_text())
    test_eq(config["project"]["dynamic"], ["version"])
    test_eq(config["tool"]["setuptools"]["dynamic"]["version"]["attr"], "pkg.__version__")
    test_eq(config["project"]["entry-points"]["nbdev"]["pkg"], "pkg._modidx:d")
    assert (project / "pkg" / "__init__.py").exists()
finally: remove_demo_path(root)